<a href="https://colab.research.google.com/github/Pgiuliano8/COMS5790-NLP-Projects/blob/main/Project_3/project3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **COMS 5790 - Project 3: RAG System for QA Task**

In [1]:

!pip install -qU langchain langchain-community langchain-huggingface sentence-transformers langchain-text-splitters faiss-cpu pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.4/170.4 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 91.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 111.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.3/573.3 kB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires panda

## 1. Data Loading & Text Pre-processing
In this section, I load the dataset and clean the raw lecture transcripts to create a unified and coherent text block.

In [2]:
import pandas as pd
import re

# 1. Load the QA test dataset
try:
    qa_test = pd.read_csv("QA_test.csv")
    print(f"Successfully loaded {len(qa_test)} questions")
except FileNotFoundError:
    print("Error: Please make sure to upload 'QA_test.csv' to the environment")

# 2. Read and clean the messy transcript text
try:
    with open("documents.txt", "r", encoding="utf-8") as f:
        raw_text = f.read()

    # Replace all newlines with spaces (since empty lines indicate pauses, not paragraphs)
    clean_text = raw_text.replace('\n', ' ')

    # Remove any multiple spaces created by the replacement
    clean_text = re.sub(r'\s+', ' ', clean_text).strip()

    print(f"Original text length: {len(raw_text)} characters.")
    print(f"Cleaned text length: {len(clean_text)} characters.")
    print("\nPreview of the cleaned text:")
    print(clean_text[:200] + "...")

except FileNotFoundError:
    print("Error: Please make sure to upload 'documents.txt' to the environment.")

Successfully loaded 110 questions
Original text length: 77504 characters.
Cleaned text length: 76034 characters.

Preview of the cleaned text:
Welcome to this lecture on animal ecology. You will learn something about toucans today. You’ve probably seen a toucan before. Maybe on a cereal box, maybe at a zoo, or in a nature documentary. Toucan...


## 2. Vector Database & Retrieval
In this section, I split the cleaned text into smaller, overlapping chunks and compute their embeddings. Then store these embeddings in a FAISS vector database to enable efficient semantic retrieval for our RAG system.

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Initialize the text splitter
# chunk size of 400 characters with an overlap of 50 characters
# to maintain context between adjacent chunks without cutting concepts in half.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ".", " ", ""]
)

# 2. Split the cleaned text into chunks
print("Splitting the cleaned text into chunks")
chunks = text_splitter.split_text(clean_text)
print(f"Total chunks created: {len(chunks)}")

# 3. Initialize the embedding model
print("\nLoading the HuggingFace embedding model")
# 'all-MiniLM-L6-v2' is a lightweight, fast, and highly effective model for semantic search
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 4. Create the FAISS Vector Database
print("Building the FAISS vector database...")
vector_db = FAISS.from_texts(chunks, embedding_model)
print("Vector database successfully built and ready for retrieval")

# 5. Set up the retriever
# configure it to return the top 3 most relevant chunks (k=3) for any given query
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

# Quick test to verify the retriever is working
sample_query = "What is a Representative?"
print(f"\nTesting retriever with a sample query: '{sample_query}'")
test_docs = retriever.invoke(sample_query)

print("\nTop retrieved chunk preview:")
print("-" * 50)
print(test_docs[0].page_content)
print("-" * 50)

Splitting the cleaned text into chunks
Total chunks created: 224

Loading the HuggingFace embedding model


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Building the FAISS vector database...
Vector database successfully built and ready for retrieval

Testing retriever with a sample query: 'What is a Representative?'

Top retrieved chunk preview:
--------------------------------------------------
. These people do not represent other people who work in the town. The key term here is represent. So you have to think about who you are asking the question to, and who you can generalize those results to. It’s very important. There’re different types of biases when you run surveys. There’s selection bias. Let’s say you go to the city at 2am and you ask people to answer some survey questions
--------------------------------------------------


## 3. LLM & Prompt Engineering
In this section, I initialize the language model and design a strict prompt template using few-shot examples to enforce the required Kaggle output format.

In [15]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


model_id = "google/flan-t5-large"
#load the tokenizer and the Seq2Seq model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id, device_map="auto")

#generation function
def generate_answer(context, question):
    prompt_template = f"""You are a highly precise question-answering assistant.
Answer the question based EXCLUSIVELY on the provided context. Ignore any real-world factual knowledge.

Strict Output Rules:
- Keep the answer as short and precise as possible (e.g., output "Cloudy" instead of "The weather tomorrow is cloudy").
- Do not use abbreviations.
- For Yes/No questions, answer ONLY "Yes" or "No".
- For fill-in-the-blank questions containing "___________", output ONLY the missing exact words.
- If the context does not contain enough information to answer, output EXACTLY "I don't know".

Examples of the exact expected output format:
Question: What represents the entire collection of objects?
Answer: Representative

Question: What does ALU stand for?
Answer: Arithmetic logic unit

Question: What represents integers in Python?
Answer: int

Question: When was the Wild Bird Conservation Act passed?
Answer: 1993

Question: What kind of laser is used?
Answer: Green laser

Now answer the following:

Context:
{context}

Question: {question}
Answer:"""

    # Tokenize input and move to the same device as the model (e.g., GPU)
    inputs = tokenizer(prompt_template, return_tensors="pt").to(model.device)

    # Generate the output text
    outputs = model.generate(**inputs, max_new_tokens=30)

    # Decode and clean the output
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer.strip()

#Quick test to verify it works
sample_context = "The toco toucan is mostly black, and it has white coloring on its throat and upper breast."
sample_question = "What color is the toco toucan mostly?"

print("\nGeneration function test")
print(f"Question: {sample_question}")
print(f"Generated Answer: {generate_answer(sample_context, sample_question)}")

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



Generation function test
Question: What color is the toco toucan mostly?
Generated Answer: black


### 4. Kaggle Submission

In [16]:
from tqdm.notebook import tqdm

predictions = []

# Iterate through all questions to generate answers
# tqdm displays a progress bar, as this will take a few minutes for 110 questions
for index, row in tqdm(qa_test.iterrows(), total=len(qa_test)):
    q_id = row['ID']
    question = row['Questions']

    #Retrieve the top 3 most relevant chunks from the FAISS database
    retrieved_docs = retriever.invoke(question)

    # Combine the retrieved chunks into a single context string
    context = " ".join([doc.page_content for doc in retrieved_docs])

    #Generate the exact answer using our custom LLM function
    answer = generate_answer(context, question)

    #Store the result in the exact format required by Kaggle
    predictions.append({
        'ID': q_id,
        'ANSWER': answer
    })

submission_df = pd.DataFrame(predictions)
submission_df.columns = ['ID', 'answer']

# Save to CSV without the pandas index
output_filename = "submission_project3.csv"
submission_df.to_csv(output_filename, index=False)

print(submission_df.head())


  0%|          | 0/110 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (513 > 512). Running this sequence through the model will result in indexing errors


   ID  answer
0   1  subset
1   2     ALU
2   3     int
3   4    1993
4   5      No
